In [71]:
import os
import ROOT
import pandas as pd

outputFolder = 'plots-combined'
os.makedirs(outputFolder, exist_ok=True)


In [72]:
# patching configuration
eff_threshold = eff_threshold
run = run

#indicative range for 45
# x_min, x_max = 2, 8
# y_min, y_max = -4, 2

#indicative range for 45
x_min, x_max = 5, 11
y_min, y_max = -2, 2


#modify if needed


In [73]:
rootFile_pattern_first = (
    '/eos/cms/store/group/dpg_ctpps/comm_ctpps/'
    'pps-automation/dev/pps-test-re-tracking-efficiency/'
    '2024postTS1_track_eff_firstbunches_v1/{run}/'
    'outputTrackingEfficiency_run{run}.root'
)

rootFile_pattern_all = (
    '/eos/cms/store/group/dpg_ctpps/comm_ctpps/'
    'pps-automation/dev/pps-test-re-tracking-efficiency/'
    '2024postTS1_track_eff_allbunches_v1/{run}/'
    'outputTrackingEfficiency_run{run}.root'
)

rootFile_first = rootFile_pattern_first.format(run=run)
rootFile_all = rootFile_pattern_all.format(run=run)


In [60]:
def fillDataFrame(csv_file):
    runs, fills, LSs, dates, del_lumis, rec_lumis = [], [], [], [], [], []

    with open(csv_file, 'r') as file:
        for line in file:
            if not line.startswith('#'):
                parts = line.strip().split(',')
                run_fill = parts[0].split(':')
                runs.append(int(run_fill[0]))
                fills.append(int(run_fill[1]))
                LSs.append(int(parts[1].split(':')[1]))
                dates.append(parts[2])
                del_lumis.append(float(parts[5]))
                rec_lumis.append(float(parts[6]))

    df = pd.DataFrame({
        'run': runs,
        'fill': fills,
        'LS': LSs,
        'date': dates,
        'del_lumi': del_lumis,
        'rec_lumi': rec_lumis
    })

    df['integrated_del_lumi'] = df['del_lumi'].cumsum()
    df['integrated_rec_lumi'] = df['rec_lumi'].cumsum()
    df['date'] = pd.to_datetime(df['date'])

    return df


def getIntegratedLumiForRun(run, df):
    lumiRun = run
    while True:
        sel = df[df['run'] == lumiRun]
        if len(sel):
            return float(sel.iloc[0]['integrated_del_lumi'])
        lumiRun += 1


In [52]:
pps_2024_track_df = fillDataFrame('pps_track_only_2024.csv')


/tmp/ipykernel_851/289081089.py:27: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['date'] = pd.to_datetime(df['date'])


In [74]:
def combine_pixel_unit_histograms(h_all, h_first):
    h_combined = h_all.Clone("h_pixel_unit_eff_combined")
    h_combined.SetDirectory(ROOT.gDirectory)

    for ix in range(1, h_combined.GetNbinsX() + 1):
        x = h_combined.GetXaxis().GetBinCenter(ix)
        if not (x_min < x < x_max):
            continue

        for iy in range(1, h_combined.GetNbinsY() + 1):
            y = h_combined.GetYaxis().GetBinCenter(iy)
            if not (y_min < y < y_max):
                continue

            if h_first.GetBinContent(ix, iy) < eff_threshold:
                h_combined.SetBinContent(ix, iy, h_first.GetBinContent(ix, iy))

    return h_combined


In [75]:
import utils
def makeFancyPatchedPlot(run, sector, station, z_max=0.3):

    rootFile_all = rootFile_pattern_all.format(run=run)
    rootFile_first = rootFile_pattern_first.format(run=run)

    intLumi = getIntegratedLumiForRun(run, pps_2024_track_df)

    pattern = (
        'DQMData/Run 999999/Run summary/'
        'h2RefinedTrackEfficiency_arm{arm}_st{station}_rp3'
    )

    h_all = utils.getPlot(rootFile_all, sector, station, pattern)
    h_first = utils.getPlot(rootFile_first, sector, station, pattern)

    hist = combine_pixel_unit_histograms(h_all, h_first)
    
    plotPubStatus = 'Preliminary'
    enlarge_factor = 1.5

    canvas = ROOT.TCanvas(
        'c', 'c',
        round(104*6*enlarge_factor),
        round(160*4*enlarge_factor)
    )

    # --- drawing (identical style) ---
#     canvas = ROOT.TCanvas('c', 'c', 1000, 800)
    right_margin = 0.15
    canvas.SetRightMargin(right_margin)
    
    ROOT.gStyle.SetOptStat(0)
    ROOT.gStyle.SetOptTitle(0)
    ROOT.gStyle.SetPalette(1)

    hist.SetMinimum(z_max)
    hist.GetXaxis().SetRangeUser(0, 19)
    hist.GetYaxis().SetRangeUser(-8, 16)
    hist.GetZaxis().SetTitle('Efficiency')
    hist.GetZaxis().SetTitleOffset(1.30)

    hist.Draw("COLZ")

    st_m = '220' if station == '2' else '210'
    latex = ROOT.TLatex()


    cmsText = '#font[61]{CMS} #scale[0.76]{#font[52]{'+plotPubStatus+'}}'
    stationTag = f'#scale[0.76]{{#font[42]{{{sector}-{st_m}-fr}}}}'
    year_energy_tag = '#scale[0.76]{#font[42]{2024 (13.6 TeV)}}'

    latex.SetTextAlign(11)
    latex.DrawLatexNDC(0.15, 0.85, cmsText)

    latex.SetTextAlign(31)
    latex.DrawLatexNDC(1-right_margin-0.01, 0.85, stationTag)
    latex.DrawLatexNDC(1-right_margin-0.01, 0.91, year_energy_tag)

#     latex.DrawLatexNDC(0.865, 0.85, str(run))

    lumiText = f'#font[42]{{#scale[0.76]{{L = {intLumi-28:.1f} fb^{{-1}}}}}}'
    latex.DrawLatexNDC(1-right_margin-0.01, 0.13, lumiText)

    filename = f'{outputFolder}/rad_eff_{sector}_{st_m}_run{run}_patched.png'
    canvas.SaveAs(filename)


    return canvas


In [65]:
eff_threshold = 0.3
sector = '45'
run=385842
for station in ['0','2']:
    makeFancyPatchedPlot(run, sector, station)


Info in <TCanvas::Print>: png file plots-combined/rad_eff_45_210_run385842_patched.png has been created
Info in <TCanvas::Print>: png file plots-combined/rad_eff_45_220_run385842_patched.png has been created


In [79]:
sector = '56'
run=386924
eff_threshold = 0.3
for station in ['0','2']:
    makeFancyPatchedPlot(run, sector, station)
#funziona bene solo preTS1, perche è spostato e bisogna rifinire il rettangolo indicativo.

Info in <TCanvas::Print>: png file plots-combined/rad_eff_56_210_run386924_patched.png has been created
Info in <TCanvas::Print>: png file plots-combined/rad_eff_56_220_run386924_patched.png has been created


In [ ]:
# The all-bunches efficiency map is used as the baseline.
# In the most irradiated region, bins where the first-bunch 
# efficiency drops below 0.5 are replaced with the corresponding first-bunch values, 
# in order to correctly account for ROC-induced inefficiencies.